In [13]:
using DelimitedFiles
using CairoMakie
using GLMakie
using LinearAlgebra
using Random
using CSV
using DataFrames
using Serialization

In [165]:
Randomisation = true

save_h_matrix = false
read_h_matrix = false
z_boundary_conditions = false  # if true z_boundary_conditions = true: the periodic boundary conditions are switched on. 
Interaction_radius_cutoff = false
interaction_cut_off_radius = 1e-1   # m interaction radius cut-off

visulize_sim_box = true

copy_size::Int64 = 27 # 27
nx::Int64 = 40 #40#copy_size #24 
ny::Int64 = 40 #40#copy_size #24 
nz::Int64 = 10 #10#copy_size #15

include("./introduction.jl")
include("./ir_spectra.jl")
include("./ir_spectra_h_matrix.jl")
include("./ir_spectra_centre.jl")

########################## Read the measured FTIR data ##########################
file_path_p = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_p_pol.txt")
file_p      = open(file_path_p)
header_p    = split(strip(readline(file_p)), '\t')
data_p      = readdlm(file_p, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_p = data_p[:, 1]
A_p = data_p[:, 2]
 
file_path_s = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_s_pol.txt")
file_s      = open(file_path_s)
header_s    = split(strip(readline(file_s)), '\t')
data_s      = readdlm(file_s, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_s = data_s[:, 1]
A_s = data_s[:, 2];

In [140]:
ν0 = 2136.97 # cm-1 #2050.0
νk::Vector{Float64} = collect(ν0- 1*range :step:ν0 + 1*range)
nmols_ml = 4*nx*ny*nz

if Randomisation == true    
    Random.seed!(1234)
    num = sign.(rand(nmols_ml) .- 0.5)
    eu_unit_vector = num .* eu # randomised unit vector
else
    eu_unit_vector = eu
end

64000-element Vector{Vector{Float64}}:
 [1.0, 1.0, 1.0]
 [-1.0, 1.0, 1.0]
 [-1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 ⋮
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [-1.0, 1.0, 1.0]
 [-1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0]
 [-1.0, -1.0, -1.0]
 [-1.0, 1.0, 1.0]
 [-1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0]

In [141]:
@time ipda_h_matrix, isda_h_matrix, ip_h_matrix, is_h_matrix, h_matrix = ir_spectra_h_matrix(νk, eu_unit_vector, com_ol, Δν)

Matrix{Float64}
(64000, 64000)
2
64000
1785.605124 seconds (77.82 G allocations: 3.577 TiB, 7.92% gc time, 0.02% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2136.376753485264 0.8750036072813138 … 0.00048092955414439953 0.0013732566787014783; 0.8750036072813138 2136.733517492566 … 0.0013302713504963288 0.0008435420805760763; … ; 0.00048092955414439953 0.0013302713504963288 … 2135.8649442936644 0.8750036072813138; 0.0013732566787014783 0.0008435420805760763 … 0.8750036072813138 2137.5833311006368])

In [166]:
h_matrix_path = joinpath("h-matrix", "h-matrix_" *string(nx)*"x"*string(ny)*"x"*string(nz)*"_"*"z_bound_"*string(z_boundary_conditions)*".bin")
# save h matrix as binary file
if save_h_matrix == true
    # save txt file in folder of force matrix # writedlm(h_matrix_path, h_matrix)
    # Save to a binary file
    open(h_matrix_path, "w") do io
        serialize(io, h_matrix)
        println("Matrix saved in binary format as 'matrix.bin'")
    end
end

In [167]:
# Load from binary file
if read_h_matrix == true
    h_matrix_large = open(h_matrix_path, "r") do io # takes about 1 min after restart
        deserialize(io)
    end
    # println("Matrix loaded: ", h_matrix_large)    
    # get the diagonal elements for the different energies
    h_matrix_large_energies = diag(h_matrix_large)
    fig = Figure(size=(900, 600))
    ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
    lines!(ax, h_matrix_large_energies, color=:red, label = L"energies")
    axislegend(ax, labelsize = 18, position=:lt)
    DataInspector(fig)
    display(fig)
end

In [168]:
small_box_size_fraction = 3
# get the index of each molecules and define the middle position of cuboid 
nx_centre::Int64 = nx÷(small_box_size_fraction+0.00000000001)   # get subset of alpha-CO which is smaller than small_box_size_fraction-times the size of the total simulation box (use modulo ÷ to get unit cell below exactly small_box_size_fraction the size)
ny_centre::Int64 = ny÷(small_box_size_fraction+0.00000000001) 
nz_centre::Int64 = nz

println(nx_centre,' ', ny_centre,' ', nz_centre)
nmols_centre = 4*nx_centre*ny_centre*nz_centre

13 13 10


6760

In [169]:
max_x_centre = maximum(com_ol[:, 1]) * nx_centre / nx
max_y_centre = maximum(com_ol[:, 2]) * ny_centre / ny
max_z_centre = maximum(com_ol[:, 3]) * nz_centre / nz
print(max_x_centre, ' ', max_y_centre, ' ', max_z_centre)

12.8375 12.8375 9.5

In [170]:
# indexing of molecules in the centre of the simulation box:
com_ol_centre         = zeros(Float64, nmols_centre, 3)
eu_unit_vector_centre = Vector{Vector{Float64}}(undef, nmols_centre)
large_h_matrix_sum = zeros(Float64, nmols_centre)

count = 1
for row_i in 1:size(com_ol, 1)
    if com_ol[row_i, 1] <= max_x_centre && com_ol[row_i, 2] <= max_y_centre && com_ol[row_i, 3] <= max_z_centre
        com_ol_centre[count, 1] = com_ol[row_i, 1]
        com_ol_centre[count, 2] = com_ol[row_i, 2]
        com_ol_centre[count, 3] = com_ol[row_i, 3]
        eu_unit_vector_centre[count] = eu_unit_vector[row_i]
        large_h_matrix_sum[count] = h_matrix[row_i, row_i]
        count += 1
    end
end

In [176]:
if visulize_sim_box == true  # visualize the simulation box:

    X = com_ol[:, 1]
    Y = com_ol[:, 2]
    Z = com_ol[:, 3]

    U = [vec[1] for vec in eu_unit_vector]
    V = [vec[2] for vec in eu_unit_vector]
    W = [vec[3] for vec in eu_unit_vector]

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1]
    Y_centre = com_ol_centre[:, 2]
    Z_centre = com_ol_centre[:, 3]

    U_centre = [vec[1] for vec in eu_unit_vector_centre]
    V_centre = [vec[2] for vec in eu_unit_vector_centre]
    W_centre = [vec[3] for vec in eu_unit_vector_centre]

    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate", ylabel = "Y Coordinate", zlabel = "Z Coordinate")
    # scatter!(ax, , markersize=5, color=:blue)
    arrows!(ax, vec(X), vec(Y), vec(Z), 0.2 .* vec(U), 0.2 .* vec(V), 0.2 .* vec(W), arrowsize=0.15, color=:red)
    #scatter!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), markersize=30)
    arrows!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), 0.2 .* vec(U_centre), 0.2 .* vec(V_centre), 0.2 .* vec(W_centre), arrowsize=0.15, color=:blue)

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [172]:
# read the force matrix for the maximum size and do eigen(h) operation of small simulation box with entries of the large h matrix on the diagonal
@time ipda, isda, ip, is, eigenvecs = ir_spectra_centre(νk, eu_unit_vector_centre,  com_ol_centre, Δν, nmols_centre, large_h_matrix_sum)

6760Matrix{Float64}
(6760, 6760)
2
6760
 36.949041 seconds (926.15 M allocations: 42.067 GiB, 5.40% gc time, 0.44% compilation time)


([1.8804198823503248e-102, 2.7444515807927287e-100, 3.78949864958068e-98, 4.950328340819871e-96, 6.118049561908271e-94, 7.15351414333846e-92, 7.913236413978607e-90, 8.281670479078592e-88, 8.199960906425054e-86, 7.68133402913347e-84  …  0.6413857259594886, 0.6095376674971998, 0.5719947170651359, 0.5310080742111454, 0.48914951185271816, 0.44901961704720883, 0.41296031397751465, 0.3828145462254979, 0.35975688793017735, 0.3442061775137516], [2.2068304841318387e-102, 3.2208216619746925e-100, 4.447228486428501e-98, 5.809487695574764e-96, 7.179802051751433e-94, 8.394870603518162e-92, 9.2863094912666e-90, 9.718532204596559e-88, 9.622488598980442e-86, 9.013723342465507e-84  …  0.4091392875829839, 0.3775257099125875, 0.3423608065625907, 0.3067986983261583, 0.2738451907979469, 0.24605503057435368, 0.22536428313925455, 0.21303547805653217, 0.2096658037848673, 0.2152096140194109], [3.325045142305049e-102, 4.852148244770419e-100, 6.698710714032053e-98, 8.749219184926423e-96, 1.0811067331868906e-93, 

In [173]:
α = 0*degrees 

ipda_α = (cos(α))^2 .* ipda + (sin(α))^2 .* isda 
isda_α = (cos(α))^2 .* isda + (sin(α))^2 .* ipda;

In [174]:
# CairoMakie.activate!()
# GLMakie.activate!()
fig = Figure(size=(900, 600))

ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

conversion_to_mOD = 1000

lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
lines!(ax, νk, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
lines!(ax, νk, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")
#lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
#lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig)

GLMakie.Screen(...)